In [3]:
import numpy as np
import matplotlib.pyplot as plt
import time

NZ = 100
Ny = 1
shape    = (Ny, Nz)    # Nombre de píxels (Ny, Nz) a la pantalla.
y_range  = (0, 0) # Rang en el pla equatorial del forat negre.
z_range  = (0, 10 ) # Rang en la direcció de l'eix de rotació.
M        = 1           # Massa del forat negre.

r_horizon = 2*M
r_shadow = 3*np.sqrt(3)*M
r_ISCO = 6*M
r_disc = 12  # 18

# Posem el paràmetre afí lam = 0 per a tots els fotons.
lam, dlam = 0, 0.01

# Angle de 0 a 2pi, que serà útil per dibuixar cercles.
contour  = np.linspace(0, 2 * np.pi, 100)



In [4]:
# Definim les coordenades (t, x, y, z) inicials de l'eixam. 
t_screen = 0.0
x_screen = 20.0

y_screen = np.linspace(*y_range, Ny) [:,None]
z_screen = np.linspace(*z_range, Nz) [None,:]


# Passem a coordenades esfèriques, per la mètrica de Schwarzschild.
t  = t_screen
r  = np.sqrt(x_screen**2 + y_screen**2 + z_screen**2)
th = np.arccos(z_screen / r)
ph = np.arctan2(y_screen, x_screen)

# Posem primer les components espacials de la 4-velocitat, alineada amb l'eix x.
# Aquestes expressions són les components de - d/dx en coordenades esfèriques.
dr_dl  = - np.sin(th) * np.cos(ph)
dth_dl = - np.cos(th) * np.cos(ph) / r
dph_dl = + np.sin(ph) / (r * np.sin(th))

# Ara normalitzem per tal que dt / dlambda = 1 i la 4-velocitat sigui nul·la.
norm2  = dr_dl**2 / (1 - 2 * M / r) + r**2 * (dth_dl**2 + np.sin(th)**2 * dph_dl**2)
factor = np.sqrt(norm2 / (1 - 2*M / r))

# Definim un eixam de fotons sobre la pantalla, amb els que farem el ray tracing.
swarm = np.zeros(shape + (8,))

swarm[..., 0] = t
swarm[..., 1] = r
swarm[..., 2] = th
swarm[..., 3] = ph

swarm[..., 4] = 1
swarm[..., 5] = dr_dl  / factor
swarm[..., 6] = dth_dl / factor
swarm[..., 7] = dph_dl / factor

# Comprovem que la 4-velocitat és efectivament nul·la
np.allclose(- (1 - 2 * M / r) * swarm[..., 4]**2 + swarm[..., 5]**2 / (1 - 2 * M / r) + r**2 * (swarm[..., 6]**2 + np.sin(th)**2 * swarm[..., 7]**2), 0)

True

In [5]:
# Aquí podem visualitzar les coordenades...
'''
fig, ax = plt.subplots(1, 1, figsize = (12, 8))
plt.rc('font', size = 16)

ish = ax.imshow(r.T, cmap = 'hot', origin = 'lower', extent = y_range + z_range)
fig.colorbar(ish, ax=ax, location='right', anchor=(0, 0.3), shrink=0.8)

print(r[0,0])
'''

"\nfig, ax = plt.subplots(1, 1, figsize = (12, 8))\nplt.rc('font', size = 16)\n\nish = ax.imshow(r.T, cmap = 'hot', origin = 'lower', extent = y_range + z_range)\nfig.colorbar(ish, ax=ax, location='right', anchor=(0, 0.3), shrink=0.8)\n\nprint(r[0,0])\n"

In [6]:
# Càlcul dels símbols de Christoffel, que es retornen amb la forma (Ny, Nz).
# Important: cal tenir en compte que són simètrics en els dos índexs de baix.
def christoffel(swarm, indices):
    # No necessitem la 4-velocitat per als Christoffels, la podem llençar
    _, r, th, ph, _, _, _, _  = swarm.transpose(2, 0, 1)

    if indices == (1, 0, 0):  # Γ^r_tt
        gamma = M * (r - 2 * M) / r**3

    elif (indices == (0, 0, 1)) or (indices == (0, 1, 0)):  # Γ^t_tr
        gamma = M / (r * (r - 2 * M))

    elif indices == (1, 1, 1):  # Γ^r_rr
        gamma = -M / (r * (r - 2 * M))

    elif indices == (1, 2, 2):  # Γ^r_θθ
        gamma = -(r - 2 * M)

    elif indices == (1, 3, 3):  # Γ^r_φφ
        gamma = -(r - 2 * M) * np.sin(th)**2

    elif (indices == (2, 1, 2)) or (indices == (2, 2, 1)):  # Γ^θ_rθ
        gamma = 1 / r

    elif indices == (2, 3, 3):  # Γ^θ_φφ
        gamma = -np.sin(th) * np.cos(th)

    elif (indices == (3, 1, 3)) or (indices == (3, 3, 1)):  # Γ^φ_rφ
        gamma = 1 / r

    elif (indices == (3, 2, 3)) or (indices == (3, 3, 2)):  # Γ^φ_θφ
        gamma = np.cos(th) / np.sin(th)

    else:
        gamma = None

    return gamma


In [7]:

# Tasca 1: Cal implementar aquí una funció que calculi la derivada de l'array
# swarm respecte al paràmetre afí a partir de l'equació geodèsica.
def geoeq(swarm_in, mask):
    
    dswarm = np.zeros_like(swarm) 
    dswarm[..., :4] = swarm[..., 4:]
    
    for idx1 in range(4):
        for idx2 in range(4):
            for idx3 in range(idx2, 4):
                gamma = christoffel(swarm, (idx1, idx2, idx3))
                if gamma is not None:
                    # Sumem dues vegades els termes amb índexs diferents, per la simetria dels símbols de Christoffel.
                    dswarm[..., idx1 + 4] -= (2 - (idx2 == idx3)) * gamma * swarm[..., idx2 + 4] * swarm[..., idx3 + 4]

    
    dswarm[stopped_mask, :] = 0.0
    
    return dswarm

# Tasca 2: Cal implementar un integrador Runge-Kutta 4 que aprofiti la funció
# anterior per evolucionar l'array swarm un pas de paràmetre afí dlam.

def RK4(swarm0, h, mask):
    hh = h / 2.0
    
    k1 = geoeq(swarm0,mask)
    k2 = geoeq(swarm0 + hh * k1, mask)
    k3 = geoeq(swarm0 + hh * k2, mask)
    k4 = geoeq(swarm0 + h * k3, mask)
    
    swarm1 = swarm0 + (h / 6.0) * (k1 + 2.0*k2 + 2.0*k3 + k4)
    
    return swarm1



In [ ]:
#Càlcul
t_inici = time.time()


#Defineixo els parametres de la integració

                #ja s'ha definit el pas, dlam.
steps =1000        #m'interessa més triar el nombre de passos d'integració es fan que no el valor final del parametre afí


mask = np.ones(shape)
mask1 = np.zeros(shape)

for n in range(steps):

    swarm1 = RK4(swarm, dlam, mask)

    
# Miro si el radi es menor al radi de Schwarzschild o si han creuat el disc (si hi ha un canvi de signe en la z)

    r = swarm1[...,1]
    z0 = swarm[...,1]*np.cos(swarm[...,2])
    z1 = swarm1[...,1]*np.cos(swarm1[...,2])

    for i in range(Nz):
        for j in range(Ny):
    
            if r[i,j]<2*M:    # o arriba al BH (r = 2M) 
                mask[i, j] = 0
                mask1[i,j] = 1
            if r[i,j]<6*M and z0[i,j]*z1[i,j]<0: #o creua el disc d'acreció (radi maxim = r_ISCO = 6M)
                mask[i, j] = 0
                mask1[i,j] = 2

    swarm = swarm1.copy()

t_final = time.time()
np.savetxt("simulacio_resultats.txt", mask, delimiter=",", header="Dades de la simulació")

#dades_carregades = np.loadtxt("simulacio_resultats.txt", delimiter=",")

print(mask)


temps_total = t_final - t_inici
print(f"La simulació ha trigat: {temps_total:.4f} segons")


In [ ]:
#Plotejem el resultat (IA)

# --- Creem un colormap personalitzat ---
# Això diu: valor 0 -> vermell, valor 1 -> negre
# Cal crear una llista de colors.
# El primer color (index 0) serà el color per al valor 0, etc.
colors = ['black', 'red', 'green']
cmap = plt.cm.colors.ListedColormap(colors)

# --- Fem el plot ---

# La funció imshow és la clau
# `cmap` assigna els colors
# `interpolation='nearest'` fa que els "píxels" es vegin quadrats i nítids
plt.imshow(mask1, cmap=cmap, interpolation='nearest', extent = y_range + z_range)
plt.title('Representació de la mask (en vermell son els fotos que han "xocat")')

# Ajusta els marges per eliminar l'espai en blanc
plt.tight_layout()

plt.plot(2 * M * np.cos(contour), 2 * M * np.sin(contour), '--k', color='blue')

# Mostrem el plot
plt.show()

In [ ]:
# Definim una figura per poder representar imatges
###############################################################################
'''
fig, ax = plt.subplots(1, 1, figsize = (8, 8))
plt.rc('font', size = 16)

# Per exemple, representem un símbol de Christoffel sobre la pantalla.
G = christoffel(swarm, (3, 2, 3))

ax.set_xlabel('y')
ax.set_ylabel('z')
ax.plot(2 * M * np.cos(contour), 2 * M * np.sin(contour), '--k')
ax.text(-1.0, 2.5, 'r = 2M')
ax.imshow(G.T, cmap = 'hot', origin = 'lower', extent = y_range + z_range);
'''

In [ ]:
#buscar on divideix per sin th pk pot ser 0, tot i això es soluciona augmentan el dl = 0.01
